# Trip Concierge — End-to-End Evaluation

Replay every labeled turn in `trip_conversations.json` through the **whole concierge pipeline**
(`get_concierge_response`) and measure how well the Trip Agent + Advisors pick the right action.

**What we measure**
- Overall **accuracy** across all labeled turns.
- A **4×4 confusion matrix** — true action vs. predicted action.
- Per-class **precision / recall / F1**. `continue` dominates the dataset, so the per-class view keeps
  weak terminal-action performance (`book` / `abandon`) from hiding behind a high overall number.

**How it works.** Every `concierge` turn in the dataset carries a gold `label`. For each one we feed the
conversation *history up to that point* into `get_concierge_response(...)` and compare the action it
returns against the gold label. The result is the **baseline** that Step 6 (prompt tuning) and Step 7
(fine-tune) are measured against.

> ⚠️ This notebook calls the OpenAI API once per labeled turn (~34 calls). It costs a little and takes
> a minute or two end to end.

## 1. Imports & project path

Pull in the standard libraries, then anchor the notebook to the **repo root** in two ways:

1. **`os.chdir(REPO_ROOT)`** — the app opens `data/packages.db` and `data/chroma_db/` using paths
   relative to the *current working directory*. The kernel starts in `tests/`, so without this the
   Budget Advisor's SQLite lookup fails with `unable to open database file`. Changing into the repo
   root makes those relative paths resolve.
2. **`sys.path.insert(0, REPO_ROOT)`** — so `from app.main import ...` can find the `app` package.

This notebook lives in `tests/`, so the repo root is one directory up (computed re-run-safely, since
after the `chdir` the working directory is already the repo root).

In [1]:
import json
import os
import sys
from pathlib import Path
from collections import Counter
from datetime import date

import pandas as pd

# This notebook lives in tests/, but the app opens its data with paths relative to the
# REPO ROOT (e.g. data/packages.db, data/chroma_db). Compute the repo root re-run-safely
# (after the chdir below, cwd is already the repo root, so guard on the folder name), then
# chdir into it so those relative paths resolve no matter where the kernel started.
_here = Path.cwd()
REPO_ROOT = _here.parent if _here.name == "tests" else _here
os.chdir(REPO_ROOT)

# Put the repo root on sys.path so `from app.main import ...` resolves.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## 2. Load environment variables

Load your `.env` (`OPENAI_API_KEY`, and optionally `BOOKING_ADVISOR_MODEL`). This **must** run *before*
the next cell, because importing the app builds the `ChatOpenAI` clients at import time — if the key
isn't in the environment yet, that import raises.

In [2]:
from dotenv import load_dotenv

# Load OPENAI_API_KEY (and optional BOOKING_ADVISOR_MODEL) before importing the app below —
# the agents build their ChatOpenAI clients at import time and need the key present.
load_dotenv(REPO_ROOT / ".env")

# Confirm the key landed in the environment, without printing the secret itself.
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


## 3. Import the concierge entry point

With the environment loaded, import the single function the whole app routes through. Importing it also
constructs the Trip Agent and all three advisors, so keep this **after** the `.env` load above.

In [3]:
# The single entry point the whole pipeline routes through (Trip Agent -> Advisors).
from app.main import get_concierge_response

## 4. Load the labeled dataset

Read `trip_conversations.json`. We need two things out of it:
- `reference_date` — the fixed "today" we pass into the pipeline so date-anchored cases (the nothing-in-budget
  edge, the soft date anchor) stay **deterministic**.
- `conversations` — the list of labeled conversations.

`get_concierge_response` expects a `datetime.date`, so convert the date string with `date.fromisoformat`.

In [4]:
DATA_PATH = REPO_ROOT / "tests" / "trip_conversations.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

conversations = data["conversations"]
# Parse to a date object (get_concierge_response takes a date). Pinning it keeps the
# date-anchored cases (nothing-in-budget, soft date anchor) deterministic across runs.
REFERENCE_DATE = date.fromisoformat(data["reference_date"])

print(f"Number of conversations = {len(conversations)}")
print(f"REFERENCE_DATE = {REFERENCE_DATE}")

Number of conversations = 14
REFERENCE_DATE = 2026-06-24


## 5. Flatten conversations into eval cases

Each labeled `concierge` turn becomes one **eval case**:
- its **input** is the conversation history *before* that turn, and
- its **target** (`gold`) is the turn's `label`.

We map the dataset's speakers to the OpenAI-style roles the app expects: **`traveller → user`**,
**`concierge → assistant`**. Walk each conversation turn by turn, accumulating `history`. The key
ordering rule: **snapshot the history *before* appending the current turn**, so a concierge turn's own
text is part of the history for the *next* case but never its own input. The last message in every
snapshot is the traveller line the concierge is responding to.

In [5]:
# Dataset speaker -> the message role get_concierge_response expects.
ROLE_OF = {"traveller": "user", "concierge": "assistant"}

cases = []  # each: {conversation_id, turn_id, scenario, history: [{role, content}], gold}

for conv in conversations:
    history = []
    for turn in conv["turns"]:
        # Each concierge turn becomes one eval case: its input is the history BEFORE it, its
        # target is its gold label. Snapshot before appending so the turn being predicted is
        # never part of its own input.
        if turn["speaker"] == "concierge":
            cases.append({
                "conversation_id": conv["conversation_id"],
                "turn_id": turn["turn_id"],
                "scenario": conv["scenario"],
                "history": list(history),  # copy — later appends must not mutate this snapshot
                "gold": turn["label"],
            })
        # Every turn (both speakers) joins the running history, after any snapshot above.
        history.append({"role": ROLE_OF[turn["speaker"]], "content": turn["text"]})

print(len(cases), "eval cases |", dict(Counter(c["gold"] for c in cases)))

34 eval cases | {'continue': 16, 'recommend': 9, 'book': 5, 'abandon': 4}


## 6. Sanity-check one case

Before spending any API calls, eyeball a single case to confirm the slice is correct: the **last**
message should be the traveller line the concierge is about to respond to, and `gold` should be the
action you'd expect. Pick a non-`continue` case so the slice is interesting.

In [6]:
# Eyeball one non-`continue` case: the last message should be the traveller line the concierge
# is responding to, and the gold label should match what we'd expect.
example = next(c for c in cases if c["gold"] == "book")
print(example["scenario"], "->", example["gold"])
for m in example["history"]:
    print(f'  {m["role"]:9} | {m["content"][:80]}')

happy path: continue -> recommend -> book (Bali, no budget) -> book
  user      | Hi! I'd love some help planning a trip.
  assistant | Happy to help you plan! Where are you thinking of going, and roughly when? If yo
  user      | I've settled on Bali. Can you show me what packages you have?
  assistant | Here are the best available Bali packages I found:
1. Bali Stay 3 — depart Feb 1
  user      | Let's book option 1.


## 7. Run the pipeline on every case

Loop through the cases and call `get_concierge_response(history, REFERENCE_DATE)` for each. It returns
`(action, reply)` — keep the **action** as the prediction. Wrap each call in `try/except` so one bad
turn doesn't abort the whole run. The agents run at `temperature=0`, so re-runs are fairly stable (not
100% identical, but close).

> This is the cell that hits the API (~34 calls).

In [7]:
y_true, y_pred = [], []

# Replay every case through the full pipeline. get_concierge_response returns (action, reply);
# we keep the action as the prediction. The try/except records a failed turn as an "ERROR: ..."
# string instead of aborting the whole run, so one bad turn doesn't cost us the other 33.
for i, case in enumerate(cases):
    try:
        action, _reply = get_concierge_response(case["history"], REFERENCE_DATE)
    except Exception as e:
        action = f"ERROR: {e}"
    case["pred"] = action
    y_true.append(case["gold"])
    y_pred.append(action)
    print(f'[{i+1:>2}/{len(cases)}] gold={case["gold"]:9} pred={action:9} | {case["scenario"][:45]}')

[ 1/34] gold=continue  pred=continue  | happy path: continue -> recommend -> book (Ba
[ 2/34] gold=recommend pred=recommend | happy path: continue -> recommend -> book (Ba
[ 3/34] gold=book      pred=continue  | happy path: continue -> recommend -> book (Ba
[ 4/34] gold=continue  pred=continue  | happy path with destination question: continu
[ 5/34] gold=recommend pred=recommend | happy path with destination question: continu
[ 6/34] gold=book      pred=book      | happy path with destination question: continu
[ 7/34] gold=continue  pred=continue  | abandon: walks away early without booking
[ 8/34] gold=abandon   pred=abandon   | abandon: walks away early without booking
[ 9/34] gold=recommend pred=recommend | EDGE musing-then-booking: recommend -> contin
[10/34] gold=continue  pred=continue  | EDGE musing-then-booking: recommend -> contin
[11/34] gold=book      pred=book      | EDGE musing-then-booking: recommend -> contin
[12/34] gold=continue  pred=continue  | EDGE out-of-set questi

## 8. Overall accuracy

The headline number: the fraction of turns where the predicted action matched the gold label.

In [8]:
# Fraction of turns whose predicted action matched the gold label (errored turns count as misses).
correct = sum(t == p for t, p in zip(y_true, y_pred))
print(f"accuracy: {correct / len(y_true):.1%}  ({correct}/{len(y_true)})")

accuracy: 91.2%  (31/34)


## 9. Confusion matrix (4×4)

Rows = the **true** action, columns = what the pipeline **predicted**. The diagonal is correct;
off-diagonal cells reveal *which* actions get confused — e.g. `book → continue` means the Booking
Advisor demoted a genuine commitment; `recommend → continue` means a Budget Advisor veto. We fix the
label order so the grid always reads the same way.

In [9]:
from sklearn.metrics import confusion_matrix

# Fixed label order so the grid always reads the same way; rows = true, cols = predicted.
ACTIONS = ["continue", "recommend", "book", "abandon"]

# An errored prediction isn't one of ACTIONS, so it drops out of the grid (still counted as a
# miss in the accuracy above) — the matrix can sum to fewer than len(cases).
cm = confusion_matrix(y_true, y_pred, labels=ACTIONS)
display(pd.DataFrame(cm, index=ACTIONS, columns=ACTIONS))

,continue,recommend,book,abandon
continue,16,0,0,0
recommend,0,9,0,0
book,3,0,2,0
abandon,0,0,0,4


## 10. Per-class precision / recall / F1

For each action:
- **precision** — of the turns we *predicted* X, how many were really X (false-alarm control),
- **recall** — of the turns that *were* X, how many we caught (miss control),
- **F1** — their harmonic mean.

This is where weak terminal-action performance shows up even when overall accuracy looks fine.

In [10]:
from sklearn.metrics import classification_report

# Per-class precision / recall / F1. zero_division=0 reports 0.0 (instead of raising) for a
# class the pipeline never predicted. This is where weak book / abandon performance shows up.
report = classification_report(y_true, y_pred, labels=ACTIONS, output_dict=True, zero_division=0)
display(pd.DataFrame(report).T)

,precision,recall,f1-score,support
continue,0.842105,1.000000,0.914286,16.000000
recommend,1.000000,1.000000,1.000000,9.000000
book,1.000000,0.400000,0.571429,5.000000
abandon,1.000000,1.000000,1.000000,4.000000
accuracy,0.911765,0.911765,0.911765,0.911765
macro avg,0.960526,0.850000,0.871429,34.000000
weighted avg,0.925697,0.911765,0.896639,34.000000


## 11. Inspect the misclassifications

List every turn we got wrong, with its scenario and the last traveller message. This is the single most
useful output for **Step 6**: it tells you exactly which prompts / few-shot examples to add (e.g. lots
of `book → continue` misses ⇒ the Booking Advisor is too conservative).

In [11]:
# Every turn we got wrong, with the last traveller line that drove it — the most useful view
# for step 6 (e.g. lots of book -> continue means the Booking Advisor is too cautious).
misses = [
    {
        "conv": c["conversation_id"], "turn": c["turn_id"], "scenario": c["scenario"],
        "gold": c["gold"], "pred": c["pred"],
        "last_user": next((m["content"] for m in reversed(c["history"]) if m["role"] == "user"), ""),
    }
    for c in cases if c.get("pred") != c["gold"]
]

display(pd.DataFrame(misses))

,conv,turn,scenario,gold,pred,last_user
0,1,6,happy path: continue -> recommend -> book (Bal...,book,continue,Let's book option 1.
1,7,6,EDGE nothing-in-budget: recommend -> continue ...,book,continue,The $429 one sounds perfect — book it.
2,11,8,full path with suggestion + grounded Q: contin...,book,continue,Book option 1.


## 12. (Optional) Save a baseline snapshot

Stash the numbers so you can compare against them after prompt tuning and fine-tuning. Keep it light —
a tiny JSON file, or just jot the accuracy into `PLAN.md` next to Step 5.

In [12]:
# Save the baseline so later runs (after prompt tuning / fine-tuning) can be compared against it.
summary = {
    "accuracy": correct / len(y_true),
    "n": len(y_true),
    "by_class": pd.DataFrame(report).T.to_dict() if "report" in dir() else None,
}
(REPO_ROOT / "tests" / "baseline_metrics.json").write_text(json.dumps(summary, indent=2))

993